In [1]:
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/Research Code and Data/"
PROCESSED_PATH = BASE_PATH + "data_processed/"

Mounted at /content/drive


In [2]:
import pandas as pd

thar  = pd.read_csv(PROCESSED_PATH + "thar_standardized.csv")
hasoc = pd.read_csv(PROCESSED_PATH + "hasoc_standardized.csv")
aold  = pd.read_csv(PROCESSED_PATH + "aold_standardized.csv")
combined = pd.read_csv(PROCESSED_PATH + "combined_standardized.csv")

In [3]:
import re
import string

In [6]:
# Reload fresh standardized HASOC
hasoc = pd.read_csv(PROCESSED_PATH + "hasoc_standardized.csv")

# Apply cleaning only
temp = hasoc.copy()
temp['text'] = temp['text'].apply(clean_text)

# Count empty rows
empty_rows = (temp['text'].str.strip() == "").sum()

print("Empty rows after cleaning:", empty_rows)

# Count duplicates after cleaning
duplicates_after_clean = temp.duplicated(subset=['text']).sum()

print("Duplicates after cleaning:", duplicates_after_clean)

print("Original:", len(hasoc))
print("After cleaning (before removing anything):", len(temp))

Empty rows after cleaning: 3
Duplicates after cleaning: 900
Original: 7593
After cleaning (before removing anything): 7593


In [7]:
def clean_text(text):

    text = str(text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtag symbol only
    text = re.sub(r'#', '', text)

    # Keep multilingual characters
    text = re.sub(r'[^\w\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [8]:
datasets = {
    "thar": thar,
    "hasoc": hasoc,
    "aold": aold,
    "combined": combined
}

for name, df in datasets.items():

    print(f"\nCleaning {name}...")

    # Clean text
    df['text'] = df['text'].apply(clean_text)

    # Remove empty rows
    df = df[df['text'].str.strip() != ""]

    # Remove duplicates based on text
    df = df.drop_duplicates(subset=['text'])

    # Reset index
    df = df.reset_index(drop=True)

    datasets[name] = df

    print("Final shape:", df.shape)


Cleaning thar...
Final shape: (11546, 2)

Cleaning hasoc...
Final shape: (6692, 2)

Cleaning aold...
Final shape: (8061, 2)

Cleaning combined...
Final shape: (26299, 3)


In [9]:
for name, df in datasets.items():
    print(f"\n{name} label distribution")
    print(df['label'].value_counts())


thar label distribution
label
0    6092
1    5454
Name: count, dtype: int64

hasoc label distribution
label
0    3393
1    3299
Name: count, dtype: int64

aold label distribution
label
0    5679
1    2382
Name: count, dtype: int64

combined label distribution
label
0    15164
1    11135
Name: count, dtype: int64


In [10]:
datasets["thar"].to_csv(PROCESSED_PATH + "thar_cleaned.csv", index=False)
datasets["hasoc"].to_csv(PROCESSED_PATH + "hasoc_cleaned.csv", index=False)
datasets["aold"].to_csv(PROCESSED_PATH + "aold_cleaned.csv", index=False)
datasets["combined"].to_csv(PROCESSED_PATH + "combined_cleaned.csv", index=False)

### Data shuffle and split


In [11]:
thar  = pd.read_csv(PROCESSED_PATH + "thar_cleaned.csv")
hasoc = pd.read_csv(PROCESSED_PATH + "hasoc_cleaned.csv")
aold  = pd.read_csv(PROCESSED_PATH + "aold_cleaned.csv")
combined = pd.read_csv(PROCESSED_PATH + "combined_cleaned.csv")

#### Thar

In [12]:
thar.head()


,text,label
0,चमच ब ल द वह प क स त न म भ य सब भ जप करव रह ह ...,1
1,अब हम त म ज स औरत पर भर स नह कर सकत ह और स र श...,0
2,ea log sach bolna nhe jante zhooth per zhooth ...,0
3,insaniyat naam ki chez nhi inme,0
4,bandar bajrang khan chale bare se muh ke sath ...,1


In [15]:
SPLIT_PATH = PROCESSED_PATH + "splits/"

In [ ]:
datasets = ["thar", "hasoc", "aold","combined"]

In [14]:
from sklearn.model_selection import train_test_split

In [17]:
def split_dataset(df, dataset_name):

    print(f"\nSplitting {dataset_name}")

    # Shuffle
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

    # First split: 70% train, 30% temp
    train_df, temp_df = train_test_split(
        df,
        test_size=0.30,
        stratify=df['label'],
        random_state=42
    )

    # Second split: 15% val, 15% test
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        stratify=temp_df['label'],
        random_state=42
    )

    print("Train:", len(train_df))
    print("Validation:", len(val_df))
    print("Test:", len(test_df))

    return train_df, val_df, test_df

In [16]:
# shuffle
thar = thar.sample(frac=1, random_state=42).reset_index(drop=True)



In [18]:
thar_train, thar_val, thar_test = split_dataset(thar, "THAR")


Splitting THAR
Train: 8082
Validation: 1732
Test: 1732


In [19]:
hasoc_train, hasoc_val, hasoc_test = split_dataset(hasoc, "HASOC")


Splitting HASOC
Train: 4684
Validation: 1004
Test: 1004


In [20]:
aold_train, aold_val, aold_test = split_dataset(aold, "AOLD")


Splitting AOLD
Train: 5642
Validation: 1209
Test: 1210


In [21]:
combined_train, combined_val, combined_test = split_dataset(combined, "combined")


Splitting combined
Train: 18409
Validation: 3945
Test: 3945


In [25]:
thar_train.to_csv(SPLIT_PATH + "thar_train.csv", index=False)
thar_val.to_csv(SPLIT_PATH + "thar_val.csv", index=False)
thar_test.to_csv(SPLIT_PATH + "thar_test.csv", index=False)

In [26]:
hasoc_train.to_csv(SPLIT_PATH + "hasoc_train.csv", index=False)
hasoc_val.to_csv(SPLIT_PATH + "hasoc_val.csv", index=False)
hasoc_test.to_csv(SPLIT_PATH + "hasoc_test.csv", index=False)

In [27]:
aold_train.to_csv(SPLIT_PATH + "aold_train.csv", index=False)
aold_val.to_csv(SPLIT_PATH + "aold_val.csv", index=False)
aold_test.to_csv(SPLIT_PATH + "aold_test.csv", index=False)

In [28]:
combined_train.to_csv(SPLIT_PATH + "combined_train.csv", index=False)
combined_val.to_csv(SPLIT_PATH + "combined_val.csv", index=False)
combined_test.to_csv(SPLIT_PATH + "combined_test.csv", index=False)

In [29]:
print("\nTHAR Train Distribution")
print(thar_train['label'].value_counts(normalize=True) * 100)

print("\nTHAR Val Distribution")
print(thar_val['label'].value_counts(normalize=True) * 100)

print("\nTHAR Test Distribution")
print(thar_test['label'].value_counts(normalize=True) * 100)


THAR Train Distribution
label
0    52.759218
1    47.240782
Name: proportion, dtype: float64

THAR Val Distribution
label
0    52.771363
1    47.228637
Name: proportion, dtype: float64

THAR Test Distribution
label
0    52.771363
1    47.228637
Name: proportion, dtype: float64
